# Playground Series S6E7 — Predicting Student Health Risk
## Score: .94940

In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
DATA_DIR = Path("playground-series-s6e7")
TARGET = "health_condition"
ID_COL = "id"

rng = np.random.default_rng(SEED)
print("LightGBM", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

LightGBM 4.6.0 | pandas 3.0.3 | numpy 2.4.6


In [10]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train.shape, "| test:", test.shape)

CAT_COLS = ["diet_type", "stress_level", "sleep_quality",
            "physical_activity_level", "smoking_alcohol", "gender"]
NUM_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
            "step_count", "exercise_duration", "water_intake"]

CLASSES = ["at-risk", "unhealthy", "fit"]
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}
y = train[TARGET].map(class_to_int).to_numpy()

print("\nTarget distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))

train: (690088, 15) | test: (295753, 14)

Target distribution:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


## Feature preparation

In [3]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    X["n_missing"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)

    X["steps_per_cal"] = X["step_count"] / X["calorie_expenditure"]
    X["cal_per_min_exercise"] = X["calorie_expenditure"] / (X["exercise_duration"] + 1e-3)
    X["steps_per_min_exercise"] = X["step_count"] / (X["exercise_duration"] + 1e-3)

    for c in CAT_COLS:
        X[c] = X[c].fillna("__NA__").astype("category")

    return X


FEATURES = NUM_COLS + CAT_COLS + [
    "n_missing", "steps_per_cal", "cal_per_min_exercise", "steps_per_min_exercise",
]

X = build_features(train)[FEATURES]
X_test = build_features(test)[FEATURES]

for c in CAT_COLS:
    levels = X[c].cat.categories.union(X_test[c].cat.categories)
    X[c] = X[c].cat.set_categories(levels)
    X_test[c] = X_test[c].cat.set_categories(levels)

print("Features:", len(FEATURES))
print(X.dtypes)

Features: 17
sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
n_missing                     int64
steps_per_cal               float64
cal_per_min_exercise        float64
steps_per_min_exercise      float64
dtype: object


## Cross-validated training

In [4]:
lgb_params = dict(
    objective="multiclass",
    num_class=len(CLASSES),
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

xgb_params = dict(
    objective="multi:softprob",
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=SEED,
    n_jobs=-1,
)

cat_params = dict(
    loss_function="MultiClass",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    auto_class_weights="Balanced",
    random_seed=SEED,
    thread_count=-1,
    verbose=0,
)

X_cb = X.copy()
X_test_cb = X_test.copy()
for c in CAT_COLS:
    X_cb[c] = X_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

MODELS = ["lgb", "xgb", "cat"]
oof = {m: np.zeros((len(X), len(CLASSES))) for m in MODELS}
test_acc = {m: np.zeros((len(X_test), len(CLASSES))) for m in MODELS}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    y_tr, y_va = y[tr_idx], y[va_idx]
    sw = compute_sample_weight("balanced", y_tr)

    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(X.iloc[tr_idx], y_tr, eval_set=[(X.iloc[va_idx], y_va)],
              eval_metric="multi_logloss", categorical_feature=CAT_COLS,
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof["lgb"][va_idx] = m_lgb.predict_proba(X.iloc[va_idx])
    test_acc["lgb"] += m_lgb.predict_proba(X_test) / N_SPLITS

    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(X.iloc[tr_idx], y_tr, sample_weight=sw,
              eval_set=[(X.iloc[va_idx], y_va)], verbose=False)
    oof["xgb"][va_idx] = m_xgb.predict_proba(X.iloc[va_idx])
    test_acc["xgb"] += m_xgb.predict_proba(X_test) / N_SPLITS

    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(X_cb.iloc[tr_idx], y_tr, cat_features=CAT_COLS,
              eval_set=(X_cb.iloc[va_idx], y_va), early_stopping_rounds=100, use_best_model=True)
    oof["cat"][va_idx] = m_cat.predict_proba(X_cb.iloc[va_idx])
    test_acc["cat"] += m_cat.predict_proba(X_test_cb) / N_SPLITS

    print(f"fold {fold} done")

for m in MODELS:
    print(f"{m} OOF argmax bal-acc: {balanced_accuracy_score(y, oof[m].argmax(1)):.5f}")

oof_proba = sum(oof[m] for m in MODELS) / len(MODELS)
test_proba = sum(test_acc[m] for m in MODELS) / len(MODELS)
argmax_oof = oof_proba.argmax(axis=1)
print(f"\nBlend OOF balanced accuracy (plain argmax): {balanced_accuracy_score(y, argmax_oof):.5f}")

Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.123943
fold 0 done
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.124176
fold 1 done
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.127857
fold 2 done
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.123354
fold 3 done
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's multi_logloss: 0.125963
fold 4 done
lgb OOF argmax bal-acc: 0.94466
xgb OOF argmax bal-acc: 0.94862
cat OOF argmax bal-acc: 0.94908

Blend OOF balanced accuracy (plain argmax): 0.94883


## Tune the decision rule for balanced accuracy

In [5]:
def weighted_predict(proba: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (proba * w).argmax(axis=1)


def search_weights(proba, y_true, w_grid_u, w_grid_f, base=None):
    best_w = np.array([1.0, 1.0, 1.0]) if base is None else base.copy()
    best_score = balanced_accuracy_score(y_true, weighted_predict(proba, best_w))
    for wu in w_grid_u:
        for wf in w_grid_f:
            w = np.array([1.0, wu, wf])
            score = balanced_accuracy_score(y_true, weighted_predict(proba, w))
            if score > best_score:
                best_score, best_w = score, w
    return best_w, best_score


coarse = np.round(np.geomspace(0.5, 20, 22), 3)
best_w, best_score = search_weights(oof_proba, y, coarse, coarse)

fine_u = np.linspace(best_w[1] * 0.6, best_w[1] * 1.6, 21)
fine_f = np.linspace(best_w[2] * 0.6, best_w[2] * 1.6, 21)
best_w, best_score = search_weights(oof_proba, y, fine_u, fine_f, base=best_w)

print(f"Best weights (at-risk, unhealthy, fit): {best_w.round(3)}")
print(f"OOF balanced accuracy (tuned):   {best_score:.5f}")
print(f"OOF balanced accuracy (argmax):  {balanced_accuracy_score(y, argmax_oof):.5f}")

tuned_oof = weighted_predict(oof_proba, best_w)
print("\nConfusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:")
print(confusion_matrix(y, tuned_oof))
print("\n", classification_report(y, tuned_oof, target_names=CLASSES, digits=4))

Best weights (at-risk, unhealthy, fit): [1.    1.71  1.834]
OOF balanced accuracy (tuned):   0.94971
OOF balanced accuracy (argmax):  0.94883

Confusion matrix (rows=true, cols=pred) [at-risk, unhealthy, fit]:
[[553635  25209  13717]
 [  1761  55725    238]
 [  1790    222  37791]]

               precision    recall  f1-score   support

     at-risk     0.9936    0.9343    0.9631    592561
   unhealthy     0.6866    0.9654    0.8025     57724
         fit     0.7303    0.9495    0.8256     39803

    accuracy                         0.9378    690088
   macro avg     0.8035    0.9497    0.8637    690088
weighted avg     0.9528    0.9378    0.9417    690088



## Predict test set & write submission

In [11]:
test_pred_int = weighted_predict(test_proba, best_w)
test_pred_lbl = pd.Series(test_pred_int).map(int_to_class)

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lbl})
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv:", submission.shape)
print("\nPredicted label distribution:")
print(submission[TARGET].value_counts(normalize=True).round(4))
print("\nTrain label distribution (for reference):")
print(train[TARGET].value_counts(normalize=True).round(4))
submission.head()

Wrote submission.csv: (295753, 2)

Predicted label distribution:
health_condition
at-risk      0.8091
unhealthy    0.1171
fit          0.0738
Name: proportion, dtype: float64

Train label distribution (for reference):
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
